In [1]:
%pip install feedparser requests pandas python-dateutil newspaper3k lxml_html_clean --quiet

  DEPRECATION: Building 'tinysegmenter' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'tinysegmenter'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'feedfinder2' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'feedfinder2'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  DEPRECATION: Building 'jieba3k' using the legacy setup.py bdis

In [2]:
import time                          # Rate limiting between requests
import re                            # URL and text cleaning
import warnings
warnings.filterwarnings('ignore')    # Suppress noisy parser warnings

# --- HTTP & Feed Parsing ---
import feedparser                    # RSS/Atom feed parser
import requests                      # HTTP requests for APIs
from urllib.parse import quote_plus  # URL-encode the entity name safely
import html

# --- Date Handling ---
from dateutil import parser as date_parser  # Robust date string parsing
from datetime import datetime               # Datetime objects

# --- Data Handling ---
import pandas as pd                  # Core data structure for results

# --- Article Text Extraction (optional enrichment) ---
# newspaper3k extracts article text from a URL.
# Used here to fill `article_summary` if RSS feed has no description.
try:
    from newspaper import Article
    NEWSPAPER_AVAILABLE = True
except ImportError:
    NEWSPAPER_AVAILABLE = False
    print("[WARN] newspaper3k not available — summaries will fall back to RSS descriptions.")

print("✅ All imports successful.")

✅ All imports successful.


In [ ]:
# ── Target Entity ────────────────────────────────────────────
# Change this to any company or person name you want to screen.
entity_name = "Tesla"  # <-- EDIT THIS

# ── Source Toggles ───────────────────────────────────────────
# Set to True/False to enable or disable each news source.
USE_GOOGLE_NEWS_RSS  = True   
USE_NEWSDATA_API     = True 
USE_BING_NEWS_RSS    = False   

# ── API Keys ─────────────────────────────────────────────────
NEWSDATA_API_KEY = ""   # YOUR API KEY HERE (if using NewsData API)

# ── Retrieval Settings ───────────────────────────────────────
MAX_ARTICLES_PER_SOURCE = 20   # Cap results per source to avoid noise
REQUEST_DELAY_SECONDS   = 1    # Polite delay between HTTP requests
REQUEST_TIMEOUT_SECONDS = 10   # Timeout for any single HTTP request

# ── Output Settings ──────────────────────────────────────────
OUTPUT_CSV_PATH = f"news_{entity_name.replace(' ', '_')}_phase1.csv"

# ── Column Schema ────────────────────────────────────────────
# Defines the output DataFrame structure shared across all phases.
COLUMNS = ["title", "source", "publication_date", "article_url", "article_summary"]

print(f"✅ Configuration set. Screening entity: '{entity_name}'")
print(f"   Sources enabled → Google News RSS: {USE_GOOGLE_NEWS_RSS} | "
      f"Bing RSS: {USE_BING_NEWS_RSS} | NewsData API: {USE_NEWSDATA_API}")

✅ Configuration set. Screening entity: 'Tesla'
   Sources enabled → Google News RSS: True | Bing RSS: False | NewsData API: True


In [4]:
# ── Helper: Normalize a date string ──────────────────────────
def normalize_date(raw_date: str) -> str:
    """
    Converts any date string (RSS, ISO 8601, human-readable)
    into a consistent YYYY-MM-DD format.

    Returns 'Unknown' if the date cannot be parsed.
    """
    if not raw_date:
        return "Unknown"
    try:
        dt = date_parser.parse(raw_date, fuzzy=True)
        return dt.strftime("%Y-%m-%d")
    except Exception:
        return "Unknown"


# ── Helper: Extract article summary from URL ─────────────────
def extract_summary_from_url(url: str, fallback: str = "") -> str:
    """
    Attempts to download and parse an article's text using newspaper3k.
    Falls back to the provided `fallback` string (e.g. RSS description)
    if extraction fails or newspaper3k is unavailable.

    This is used to enrich entries where the RSS feed provides
    no description or only a partial snippet.

    Note: Some sites block scrapers; errors are silently caught.
    """
    if not NEWSPAPER_AVAILABLE:
        return fallback

    try:
        article = Article(url)
        article.download()
        article.parse()
        # Use article text (truncated) or fall back to description
        text = article.text.strip()
        if text:
            # Return first 500 characters as a preview summary
            return text[:500] + ("..." if len(text) > 500 else "")
    except Exception:
        pass  # Fail silently; fallback handles it

    return fallback


# ── Source 1: Google News RSS ─────────────────────────────────
def fetch_google_news_rss(entity: str, max_results: int = MAX_ARTICLES_PER_SOURCE) -> list:
    """
    Retrieves news articles from Google News via its public RSS feed.

    Google News RSS is completely free and requires no API key.
    It searches across thousands of news sources globally.

    Feed URL pattern:
        https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en

    Args:
        entity: The company or person name to search.
        max_results: Maximum number of articles to return.

    Returns:
        List of dicts with keys: title, source, publication_date,
        article_url, article_summary.
    """
    articles = []

    # URL-encode the entity name to handle spaces and special characters
    query_encoded = quote_plus(entity)
    rss_url = (
        f"https://news.google.com/rss/search?"
        f"q={query_encoded}&hl=en-US&gl=US&ceid=US:en"
    )

    print(f"  [Google News RSS] Fetching: {rss_url}")

    try:
        # feedparser handles RSS parsing, including malformed feeds
        feed = feedparser.parse(rss_url)

        if feed.bozo and feed.bozo_exception:
            # bozo=True means feedparser detected a malformed feed
            print(f"  [Google News RSS] Feed warning: {feed.bozo_exception}")

        for entry in feed.entries[:max_results]:
            # --- Extract source name ---
            # Google News RSS puts source in entry.source.title
            source_name = "Google News"
            if hasattr(entry, 'source') and hasattr(entry.source, 'title'):
                source_name = entry.source.title

            # --- Extract description/summary ---
            # RSS descriptions often contain HTML tags; strip them.
            raw_summary = getattr(entry, 'summary', '') or ''
            clean_summary = re.sub(r'<[^>]+>', '', raw_summary).strip()

            # If no description in RSS, attempt full article extraction
            article_url = getattr(entry, 'link', '')
            if not clean_summary and article_url:
                clean_summary = extract_summary_from_url(article_url, fallback="No summary available")

            articles.append({
                "title"           : getattr(entry, 'title', 'No Title'),
                "source"          : source_name,
                "publication_date": normalize_date(getattr(entry, 'published', '')),
                "article_url"     : article_url,
                "article_summary" : clean_summary or "No summary available",
            })

        print(f"  [Google News RSS] Retrieved {len(articles)} articles.")

    except Exception as e:
        # Catch-all: network errors, unexpected feed structures, etc.
        print(f"  [Google News RSS] ERROR: {e}")

    return articles


# ── Source 2: Bing News RSS ───────────────────────────────────
def fetch_bing_news_rss(entity: str, max_results: int = MAX_ARTICLES_PER_SOURCE) -> list:
    """
    Debug version of Bing RSS retrieval.
    """

    articles = []

    query_encoded = quote_plus(entity)
    rss_url = f"https://www.bing.com/news/search?q={query_encoded}&format=RSS"

    print(f"\n[Bing News RSS] Fetching: {rss_url}")

    try:
        response = requests.get(
            rss_url,
            timeout=REQUEST_TIMEOUT_SECONDS,
            headers={
                "User-Agent": "Mozilla/5.0",
                "Accept": "application/rss+xml, application/xml, text/xml"
            }
        )

        print(f"Status Code: {response.status_code}")
        print(f"Content Type: {response.headers.get('Content-Type')}")

        response.raise_for_status()

        # Save raw XML for inspection if needed
        with open("bing_rss_debug.xml", "w", encoding="utf-8") as f:
            f.write(response.text)

        print(f"Response Size: {len(response.text)} characters")

        feed = feedparser.parse(response.content)

        print(f"Feed Bozo: {feed.bozo}")

        if feed.bozo:
            print(f"Feed Error: {feed.bozo_exception}")

        print(f"Entries Found: {len(feed.entries)}")

        # Print first entry structure
        if len(feed.entries) > 0:
            print("\nFirst Entry Keys:")
            print(feed.entries[0].keys())

            print("\nFirst Entry:")
            print(feed.entries[0])

        for entry in feed.entries[:max_results]:

            source_name = "Bing News"

            if hasattr(entry, "source"):
                try:
                    source_name = entry.source.get("title", "Bing News")
                except Exception:
                    pass

            summary = (
                getattr(entry, "summary", "")
                or getattr(entry, "description", "")
                or "No summary available"
            )

            summary = re.sub(r"<[^>]+>", "", summary).strip()

            publication_date = normalize_date(
                getattr(entry, "published", "")
                or getattr(entry, "pubDate", "")
            )

            articles.append({
                "title": getattr(entry, "title", "No Title"),
                "source": source_name,
                "publication_date": publication_date,
                "article_url": getattr(entry, "link", ""),
                "article_summary": summary,
            })

        print(f"\n[Bing News RSS] Retrieved {len(articles)} articles.")

    except requests.exceptions.Timeout:
        print(
            f"[Bing News RSS] Request timed out after "
            f"{REQUEST_TIMEOUT_SECONDS}s."
        )

    except requests.exceptions.HTTPError as e:
        print(f"[Bing News RSS] HTTP Error: {e}")

    except Exception as e:
        print(f"[Bing News RSS] ERROR: {e}")

    return articles

# ── Source 3: NewsData.io API (Free Tier) ─────────────────────
def fetch_newsdata_api(entity: str, api_key: str, max_results: int = MAX_ARTICLES_PER_SOURCE) -> list:
    """
    Retrieves news articles from NewsData.io using their free REST API.

    Free tier: 200 requests/day, up to 10 results per request.
    Sign up at: https://newsdata.io/

    This source provides structured JSON responses with richer metadata
    than RSS feeds (e.g., category, country, language fields).

    Args:
        entity: The company or person name to search.
        api_key: Your NewsData.io API key.
        max_results: Maximum number of articles to return.

    Returns:
        List of dicts with keys: title, source, publication_date,
        article_url, article_summary.
    """
    if not api_key:
        print("  [NewsData API] Skipped — no API key provided.")
        return []

    articles = []
    endpoint = "https://newsdata.io/api/1/news"

    params = {
        "apikey"  : api_key,
        "q"       : entity,
        "language": "en",       # English articles only
        "size"    : min(max_results, 10),  # Free tier max is 10 per call
    }

    print(f"  [NewsData API] Querying for: '{entity}'")

    try:
        response = requests.get(endpoint, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()

        data = response.json()

        if data.get("status") != "success":
            print(f"  [NewsData API] API returned non-success status: {data.get('status')}")
            return []

        results = data.get("results", [])

        for item in results[:max_results]:
            # description is the article snippet; content is full text (premium)
            summary = item.get("description") or item.get("content") or "No summary available"

            articles.append({
                "title"           : item.get("title", "No Title"),
                "source"          : item.get("source_id", "NewsData"),
                "publication_date": normalize_date(item.get("pubDate", "")),
                "article_url"     : item.get("link", ""),
                "article_summary" : summary,
            })

        print(f"  [NewsData API] Retrieved {len(articles)} articles.")

    except requests.exceptions.Timeout:
        print(f"  [NewsData API] Request timed out after {REQUEST_TIMEOUT_SECONDS}s.")
    except requests.exceptions.HTTPError as e:
        print(f"  [NewsData API] HTTP error: {e}")
    except ValueError as e:
        # JSON decoding failure
        print(f"  [NewsData API] Failed to parse JSON response: {e}")
    except Exception as e:
        print(f"  [NewsData API] ERROR: {e}")

    return articles


# ── Aggregator: Merge all sources into a DataFrame ────────────
def retrieve_news(entity: str) -> pd.DataFrame:
    """
    Master orchestrator function.

    Calls all enabled sources, merges results, deduplicates
    by URL, and returns a clean pandas DataFrame.

    This is the function that downstream phases (Phase 2, 3, 4)
    will call — they import this notebook or the CSV output.

    Args:
        entity: The company or person name to screen.

    Returns:
        pd.DataFrame with columns: title, source, publication_date,
        article_url, article_summary.
        Sorted by publication_date descending (newest first).
    """
    print(f"\n🔍 Starting news retrieval for entity: '{entity}'")
    print("-" * 60)

    all_articles = []

    # --- Source 1: Google News RSS ---
    if USE_GOOGLE_NEWS_RSS:
        google_articles = fetch_google_news_rss(entity)
        all_articles.extend(google_articles)
        time.sleep(REQUEST_DELAY_SECONDS)  # Polite delay between sources

    # --- Source 2: Bing News RSS ---
    if USE_BING_NEWS_RSS:
        bing_articles = fetch_bing_news_rss(entity)
        all_articles.extend(bing_articles)
        time.sleep(REQUEST_DELAY_SECONDS)

    # --- Source 3: NewsData.io API ---
    if USE_NEWSDATA_API and NEWSDATA_API_KEY:
        newsdata_articles = fetch_newsdata_api(entity, NEWSDATA_API_KEY)
        all_articles.extend(newsdata_articles)

    print("-" * 60)

    if not all_articles:
        print("⚠️  No articles retrieved. Check your internet connection or try a different entity name.")
        return pd.DataFrame(columns=COLUMNS)

    # Build DataFrame from the list of article dicts
    df = pd.DataFrame(all_articles, columns=COLUMNS)

    # --- Deduplication ---
    # Multiple sources may return the same article URL.
    # We keep the first occurrence (preserves source diversity ordering).
    before_dedup = len(df)
    df = df.drop_duplicates(subset=["article_url"], keep="first").reset_index(drop=True)
    after_dedup = len(df)
    print(f"📋 Total articles before dedup : {before_dedup}")
    print(f"✅ Total articles after dedup  : {after_dedup} (removed {before_dedup - after_dedup} duplicates)")

    # --- Sort by date (newest first) ---
    # Articles with 'Unknown' date are pushed to the bottom.
    df['_sort_date'] = pd.to_datetime(df['publication_date'], errors='coerce')
    df = df.sort_values('_sort_date', ascending=False, na_position='last')
    df = df.drop(columns=['_sort_date']).reset_index(drop=True)

    # --- Trim whitespace across string columns ---
    for col in COLUMNS:
        df[col] = df[col].astype(str).str.strip()

    print(f"\n📰 Final article count: {len(df)}")
    return df


print("✅ All retrieval functions defined.")

✅ All retrieval functions defined.


In [5]:
# --- Execute the retrieval pipeline ---
news_df = retrieve_news(entity_name)

# --- Save to CSV for downstream phases ---
if not news_df.empty:
    news_df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"\n💾 Results saved to: '{OUTPUT_CSV_PATH}'")
    print("   → Phase 2 can load this file with: pd.read_csv('" + OUTPUT_CSV_PATH + "')")
else:
    print("\n⚠️  DataFrame is empty. No CSV saved.")


🔍 Starting news retrieval for entity: 'Tesla'
------------------------------------------------------------
  [Google News RSS] Fetching: https://news.google.com/rss/search?q=Tesla&hl=en-US&gl=US&ceid=US:en
  [Google News RSS] Retrieved 20 articles.
  [NewsData API] Querying for: 'Tesla'
  [NewsData API] Retrieved 10 articles.
------------------------------------------------------------
📋 Total articles before dedup : 30
✅ Total articles after dedup  : 30 (removed 0 duplicates)

📰 Final article count: 30

💾 Results saved to: 'news_Tesla_phase1.csv'
   → Phase 2 can load this file with: pd.read_csv('news_Tesla_phase1.csv')


In [6]:

# --- Configure pandas display settings for readable output ---
pd.set_option('display.max_colwidth', 80)       # Truncate long strings in display
pd.set_option('display.max_rows', 50)           # Show up to 50 rows
pd.set_option('display.width', 200)             # Widen terminal output

# --- Summary Statistics ---
print("=" * 70)
print(f"📊 NEWS RETRIEVAL SUMMARY — Entity: '{entity_name}'")
print("=" * 70)
print(f"Total articles retrieved : {len(news_df)}")

if not news_df.empty:
    # --- Articles per source ---
    print("\n📌 Articles by source:")
    source_counts = news_df['source'].value_counts()
    for src, count in source_counts.items():
        print(f"   {src:<40} {count} articles")

    # --- Date range coverage ---
    known_dates = news_df[news_df['publication_date'] != 'Unknown']['publication_date']
    if not known_dates.empty:
        print(f"\n📅 Date range: {known_dates.min()} → {known_dates.max()}")

    # --- Display the full DataFrame ---
    print("\n📋 Full Results DataFrame:")
    print("-" * 70)
    display(news_df)   # Uses Jupyter's rich HTML rendering

    # --- Preview: First article in detail ---
    print("\n🔎 Sample — First Article Detail:")
    print("-" * 70)
    first = news_df.iloc[0]
    for col in COLUMNS:
        value = first[col]
        # Wrap long text for readability
        if len(str(value)) > 80:
            value = str(value)[:80] + "..."
        print(f"  {col:<20}: {value}")

else:
    print("\n⚠️  No results to display.")
    print("Suggestions:")
    print("  1. Check your internet connection.")
    print("  2. Try a different entity name (e.g. 'Apple' or 'Elon Musk').")
    print("  3. Enable NEWSDATA_API with a free key from https://newsdata.io/")

print("\n" + "=" * 70)
print("✅ Phase 1 complete. DataFrame `news_df` is ready for Phase 2.")
print("=" * 70)

📊 NEWS RETRIEVAL SUMMARY — Entity: 'Tesla'
Total articles retrieved : 30

📌 Articles by source:
   insidermonkey                            3 articles
   The Motley Fool                          2 articles
   CNN                                      2 articles
   Forbes                                   1 articles
   pencitycurrent                           1 articles
   indianweb2                               1 articles
   biztoc                                   1 articles
   economictimes_indiatimes                 1 articles
   swapupdate                               1 articles
   toi                                      1 articles
   InsideEVs                                1 articles
   naijaonpoint_ng                          1 articles
   BASENOR - Tesla Accessories              1 articles
   CleanTechnica                            1 articles
   EL PAÍS English                          1 articles
   AP News                                  1 articles
   WSJ                  

,title,source,publication_date,article_url,article_summary
0,Tesla FSD Is Evolving Into A De Facto Robotaxi - Forbes,Forbes,2026-06-14,https://news.google.com/rss/articles/CBMipAFBVV95cUxNWVA0c1M0YjUwSEFmeDRvZWZ...,Tesla FSD Is Evolving Into A De Facto Robotaxi&nbsp;&nbsp;Forbes
1,"Elon Musk Is Now the World's First Trillionaire. For Tesla Shareholders, the...",The Motley Fool,2026-06-14,https://news.google.com/rss/articles/CBMimAFBVV95cUxPVldMZE9jRDEyVm0wbE1VNU5...,"Elon Musk Is Now the World's First Trillionaire. For Tesla Shareholders, the..."
2,Musk Becomes First Trillionaire: SpaceX IPO rockets his wealth past ₹91 lakh...,indianweb2,2026-06-14,https://www.indianweb2.com/2026/06/musk-becomes-first-trillionaire-spacex.html,Elon Musk has officially become the world’s first trillionaire after SpaceX’...
3,Elon Musk becomes the world's first trillionaire: He lives in a 400 square f...,toi,2026-06-14,https://timesofindia.indiatimes.com/life-style/home-garden/elon-musk-becomes...,Elon Musk has officially become the world's first trillionaire following Spa...
4,SpaceX will make me rich - in spirit,pencitycurrent,2026-06-14,"http://pencitycurrent.com/stories/spacex-will-make-me-rich-in-spirit,172585","When I was younger, I used to watch the Six Million Dollar Man with Lee Majo..."
5,Can Rivian Beat Tesla in the Long Term?,biztoc,2026-06-14,https://biztoc.com/x/d7bae8ea52076f05,They're finally here. Deliveries of the much-anticipated Rivian (NASDAQ: RIV...
6,"Morgan Stanley, BMO Capital Increase Jefferies Financial (JEF) Price Targets...",insidermonkey,2026-06-14,https://www.insidermonkey.com/blog/morgan-stanley-bmo-capital-increase-jeffe...,Jefferies Financial Group Inc. (NYSE:JEF) is included among the 10 Best Valu...
7,Lennar’s (LEN) Q2 Results Show Margin Improvement and Strong Order Activity,insidermonkey,2026-06-14,https://www.insidermonkey.com/blog/lennars-len-q2-results-show-margin-improv...,Lennar Corporation (NYSE:LEN) is included among the 10 Best Value Dividend S...
8,Telsey Advisory Raises Macy’s (M) Price Target after Solid Q1 Earnings Beat,insidermonkey,2026-06-14,https://www.insidermonkey.com/blog/telsey-advisory-raises-macys-m-price-targ...,"Macy’s, Inc. (NYSE:M) is included among the 10 Best Value Dividend Stocks to..."
9,"Elon Musk becomes world's first trillionaire, but lives in a 20x20-foot home...",economictimes_indiatimes,2026-06-14,https://economictimes.indiatimes.com/news/new-updates/elon-musk-becomes-worl...,"Elon Musk has achieved a historic milestone, becoming the world's first tril..."



🔎 Sample — First Article Detail:
----------------------------------------------------------------------
  title               : Tesla FSD Is Evolving Into A De Facto Robotaxi - Forbes
  source              : Forbes
  publication_date    : 2026-06-14
  article_url         : https://news.google.com/rss/articles/CBMipAFBVV95cUxNWVA0c1M0YjUwSEFmeDRvZWZBUXF...
  article_summary     : Tesla FSD Is Evolving Into A De Facto Robotaxi&nbsp;&nbsp;Forbes

✅ Phase 1 complete. DataFrame `news_df` is ready for Phase 2.
